In [1]:
"""
Step 5: Complete 5-Qubit Frequency-Multiplexed Readout Pipeline (Float64 + Bit-Accurate Fixed-Point).

Supports TWO modes automatically:
  Mode 1 (Real 5Q HDF5): If 'readout2019_adc.h5' is in DATA_DIR, loads the 5-transmon multiplexed ADC traces.
  Mode 2 (Multiplexed 5Q from your QICK data): Uses your existing '0528_X_train_0_770.npy' & '0528_X_test_0_770.npy'
         to modulate 5 qubits onto the exact 5 IF carriers [64.73, 25.37, -24.79, -70.27, -127.28] MHz on a shared
         ADC bus with realistic inter-resonator crosstalk (linear + nonlinear AC-Stark phase shift), so you can
         benchmark multi-qubit demuxing, crosstalk cancellation, and fixed-point RTL vectors immediately!
"""
import os
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ config
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
H5_FILE = "readout2019_adc.h5"

N_QUBITS = 5
FS = 500.0e6                                                       # 500 MSPS ADC clock (2 ns bins)
IF_FREQS = np.array([64.729e6, 25.366e6, -24.790e6, -70.269e6, -127.282e6])  # 5 resonator IFs (Hz)
T_BINS = 500                                                       # 1 us readout window (500 bins)
T_OPT = [500, 500, 420, 480, 450]                                  # per-qubit truncation windows T_k
EMA_ALPHA = 0.05
RIDGE_LAMBDA_REL = 1e-4

# Fixed-point hardware parameters (matching your Step 4 discoveries)
FP_CFG = dict(dlo_bits=18, F=17, w_bits=18, w_frac=12, clip=99.99)

N_CAL = 30_000     # multi-qubit shots for filter calibration
N_CLF = 50_000     # multi-qubit shots for Ridge classifier training
N_TEST = 20_000    # multi-qubit shots for test evaluation
N_GOLD = 500       # multi-qubit shots exported to golden_vectors_5q.npz
SEED = 42


def find_file(fname):
    for d in [DATA_DIR, os.path.join(DATA_DIR, "python"), "."]:
        cand = os.path.join(d, fname)
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(f"Cannot find {fname} in {DATA_DIR} or current folder.")


# ------------------------------------------------------------------ 1. Load or Synthesize 5-Qubit Multiplexed Data
def build_multiplexed_from_qick():
    """Constructs a 5-qubit frequency-multiplexed ADC dataset using real QICK shots + inter-qubit crosstalk."""
    print("Mode 2: Constructing 5-Qubit Frequency-Multiplexed ADC Stream from QICK dataset ...")
    Xtr_m = np.load(find_file("0528_X_train_0_770.npy"), mmap_mode="r")
    Xte_m = np.load(find_file("0528_X_test_0_770.npy"), mmap_mode="r")
    ytr_1q = np.load(find_file("0528_y_train_0_770.npy")).astype(int)
    yte_1q = np.load(find_file("0528_y_test_0_770.npy")).astype(int)

    rng = np.random.default_rng(SEED)
    t_sec = np.arange(T_BINS, dtype=np.float64) / FS
    # Complex upconversion carriers exp(+j * 2*pi * f_k * t) for each of the 5 resonators
    carriers = np.exp(1j * 2.0 * np.pi * IF_FREQS[:, None] * t_sec[None, :])  # (5, T_BINS)

    # Crosstalk coupling matrix between neighboring resonators in frequency space
    # Adjacent channels (|j-k|=1) have 14% linear leakage + 0.28 rad state-dependent AC-Stark phase pull
    lin_xtalk = np.array([
        [1.00, 0.15, 0.04, 0.01, 0.00],
        [0.15, 1.00, 0.16, 0.04, 0.01],
        [0.04, 0.16, 1.00, 0.14, 0.04],
        [0.01, 0.04, 0.14, 1.00, 0.15],
        [0.00, 0.01, 0.04, 0.15, 1.00],
    ])
    phase_xtalk = np.array([
        [0.00, 0.28, 0.08, 0.00, 0.00],
        [-0.26, 0.00, 0.30, 0.08, 0.00],
        [0.07, -0.28, 0.00, 0.26, 0.07],
        [0.00, 0.07, -0.25, 0.00, 0.28],
        [0.00, 0.00, 0.07, -0.27, 0.00],
    ])

    def synth_split(X_src, y_src, n_shots, chunk=5000):
        # Sample 5 independent QICK shots to form each 5-qubit multiplexed shot
        idx = rng.integers(0, len(y_src), size=(n_shots, N_QUBITS))
        Y_5q = y_src[idx]                                          # (n_shots, 5) binary labels in {0, 1}
        mux_I = np.empty((n_shots, T_BINS), dtype=np.int16)
        mux_Q = np.empty((n_shots, T_BINS), dtype=np.int16)

        for s in range(0, n_shots, chunk):
            e = min(s + chunk, n_shots)
            b_idx = idx[s:e]                                       # (B, 5)
            B = e - s
            # Gather baseband I and Q for the 5 qubits
            bb = np.empty((B, N_QUBITS, T_BINS), dtype=np.complex128)
            for q in range(N_QUBITS):
                # Read sorted slice chunks for fast disk access
                rows = np.asarray(X_src[b_idx[:, q], :2 * T_BINS], dtype=np.float64)
                bb[:, q, :] = rows[:, 0::2] + 1j * rows[:, 1::2]

            # 1. State-dependent nonlinear phase pull (AC-Stark / Kerr crosstalk from neighbor states)
            theta = (2.0 * Y_5q[s:e] - 1.0) @ phase_xtalk.T        # (B, 5)
            # Also add quadratic envelope distortion between adjacent excited resonators
            nl_pull = 1.0 + 0.12 * np.roll(2.0 * Y_5q[s:e] - 1.0, 1, axis=1)
            bb_coupled = bb * (nl_pull[:, :, None] * np.exp(1j * theta[:, :, None]))

            # 2. Linear resonator feedline crosstalk
            bb_coupled = np.einsum("pq,bqt->bpt", lin_xtalk, bb_coupled)

            # 3. Multiplex all 5 qubits onto the single shared IF bus: sum_k bb_k(t) * exp(j*w_k*t)
            bus = np.einsum("bkt,kt->bt", bb_coupled, carriers) / np.sqrt(N_QUBITS)

            mux_I[s:e] = np.clip(np.rint(bus.real), -32768, 32767).astype(np.int16)
            mux_Q[s:e] = np.clip(np.rint(bus.imag), -32768, 32767).astype(np.int16)

        return mux_I, mux_Q, Y_5q

    print(f"Synthesizing CAL ({N_CAL}), CLF ({N_CLF}), and TEST ({N_TEST}) 5-qubit multiplexed shots ...")
    I_cal, Q_cal, Y_cal = synth_split(Xtr_m, ytr_1q, N_CAL)
    I_clf, Q_clf, Y_clf = synth_split(Xtr_m[:450_000], ytr_1q[:450_000], N_CLF)
    I_te,  Q_te,  Y_te  = synth_split(Xte_m, yte_1q, N_TEST)
    return (I_cal, Q_cal, Y_cal), (I_clf, Q_clf, Y_clf), (I_te, Q_te, Y_te)


# Load dataset
cal_data, clf_data, te_data = build_multiplexed_from_qick()
I_cal, Q_cal, Y_cal = cal_data
I_clf, Q_clf, Y_clf = clf_data
I_te,  Q_te,  Y_te  = te_data
print(f"Multiplexed bus ready: I/Q shape={I_te.shape} (int16), Y_te shape={Y_te.shape} (5 qubits)\n")


# ------------------------------------------------------------------ 2. Stage 2: Calibration & Pre-Mixed DLO Weights
def ema_1d(x, alpha):
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = alpha * x[t] + (1.0 - alpha) * out[t - 1]
    return out


def calibrate_5q_dlo_filters(I_raw, Q_raw, Y):
    """
    For each qubit k in 0..4:
      1. Digitally demodulate raw bus by exp(-j * 2*pi * f_k * t)
      2. Compute State Matched Filter (WMF) and Algorithm 1 Relaxation Matched Filter (WRMF)
      3. Pre-mix the baseband weights with the DLO carrier exp(-j * 2*pi * f_k * t) so the FPGA
         Stage 2 MAC operates directly on the raw multiplexed (I_raw, Q_raw) stream!
    """
    t_sec = np.arange(T_BINS, dtype=np.float64) / FS
    S_raw = I_raw.astype(np.float64) + 1j * Q_raw.astype(np.float64)

    # Complex pre-mixed DLO weight tables for the 5 qubits:
    # Each filter dot-product on (I_raw, Q_raw) is:
    #   I_filt = sum_t (I_raw*Wc_I + Q_raw*Ws_I),  Q_filt = sum_t (Q_raw*Wc_Q - I_raw*Ws_Q)
    premixed_W = np.zeros((N_QUBITS, 4, 2, T_BINS), dtype=np.float64)  # (qubit, 4_feats, 2_quads[I_raw,Q_raw], T)

    for q in range(N_QUBITS):
        demod_osc = np.exp(-1j * 2.0 * np.pi * IF_FREQS[q] * t_sec)    # exp(-j*w_q*t)
        Zq = S_raw * demod_osc[None, :]                                # demodulated baseband trace
        Iq, Qq = Zq.real, Zq.imag
        yq = Y[:, q]

        # Baseband trajectories for |0> and |1>
        mI0, mQ0, vI0, vQ0 = Iq[yq == 0].mean(0), Qq[yq == 0].mean(0), Iq[yq == 0].var(0), Qq[yq == 0].var(0)
        mI1, mQ1, vI1, vQ1 = Iq[yq == 1].mean(0), Qq[yq == 1].mean(0), Iq[yq == 1].var(0), Qq[yq == 1].var(0)

        # HERQULES Algorithm 1: Unsupervised centroid labeling for |1 -> 0> relaxation traces
        mtv = np.c_[Iq.mean(1), Qq.mean(1)]
        C0, C1 = mtv[yq == 0].mean(0), mtv[yq == 1].mean(0)
        R = 0.5 * np.linalg.norm(C0 - C1)
        relax_mask = (yq == 1) & (np.linalg.norm(mtv - C0, axis=1) < R)
        mIR, mQR, vIR, vQR = Iq[relax_mask].mean(0), Qq[relax_mask].mean(0), Iq[relax_mask].var(0), Qq[relax_mask].var(0)

        print(f"Qubit {q+1} (IF={IF_FREQS[q]/1e6:+7.2f} MHz, T_opt={T_OPT[q]}): "
              f"RMF caught {relax_mask.sum():4d}/{(yq==1).sum()} relaxing |1> shots ({100*relax_mask.sum()/(yq==1).sum():.1f}%)")

        # Baseband matched filter envelopes
        w_mf_I  = ema_1d(mI1 - mI0, EMA_ALPHA) / (0.5 * (vI0 + vI1))
        w_mf_Q  = ema_1d(mQ1 - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQ1))
        w_rmf_I = ema_1d(mIR - mI0, EMA_ALPHA) / (0.5 * (vI0 + vIR))
        w_rmf_Q = ema_1d(mQR - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQR))

        # Apply per-qubit truncation window mask T_OPT[q]
        T_q = T_OPT[q]
        for w_arr in (w_mf_I, w_mf_Q, w_rmf_I, w_rmf_Q):
            w_arr[T_q:] = 0.0

        # Pre-mix with DLO carrier cos(w_q*t) and sin(w_q*t):
        # Since I_demod(t) =  I_raw(t)*cos(w_q*t) + Q_raw(t)*sin(w_q*t)
        #   and Q_demod(t) = -I_raw(t)*sin(w_q*t) + Q_raw(t)*cos(w_q*t)
        c_osc = np.cos(2.0 * np.pi * IF_FREQS[q] * t_sec)
        s_osc = np.sin(2.0 * np.pi * IF_FREQS[q] * t_sec)

        # Feature 0: a_q = sum_t w_mf_I(t) * I_demod(t)
        premixed_W[q, 0, 0, :] =  w_mf_I * c_osc
        premixed_W[q, 0, 1, :] =  w_mf_I * s_osc
        # Feature 1: b_q = sum_t w_mf_Q(t) * Q_demod(t)
        premixed_W[q, 1, 0, :] = -w_mf_Q * s_osc
        premixed_W[q, 1, 1, :] =  w_mf_Q * c_osc
        # Feature 2: c_q = sum_t w_rmf_I(t) * I_demod(t)
        premixed_W[q, 2, 0, :] =  w_rmf_I * c_osc
        premixed_W[q, 2, 1, :] =  w_rmf_I * s_osc
        # Feature 3: d_q = sum_t w_rmf_Q(t) * Q_demod(t)
        premixed_W[q, 3, 0, :] = -w_rmf_Q * s_osc
        premixed_W[q, 3, 1, :] =  w_rmf_Q * c_osc

    return premixed_W


premixed_W = calibrate_5q_dlo_filters(I_cal, Q_cal, Y_cal)


def accumulate_5q(I_raw, Q_raw, W_table, chunk=5000):
    """
    Executes Stage 2 Pre-Mixed Dual-DLO MAC across all 5 qubits simultaneously.
    Returns (N, 5, 4) accumulated scalars: [a_k, b_k, c_k, d_k] for k=0..4.
    """
    N = len(I_raw)
    out = np.empty((N, N_QUBITS, 4), dtype=np.float64)
    # Reshape W_table for fast matrix multiply: (T_BINS, 5*4)
    WI = W_table[:, :, 0, :].reshape(N_QUBITS * 4, T_BINS).T
    WQ = W_table[:, :, 1, :].reshape(N_QUBITS * 4, T_BINS).T
    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        res = I_raw[s:e].astype(np.float64) @ WI + Q_raw[s:e].astype(np.float64) @ WQ
        out[s:e] = res.reshape(e - s, N_QUBITS, 4)
    return out


# ------------------------------------------------------------------ 3. Multi-Qubit Feature Expansion & Crosstalk Cancellation
def build_multiqubit_features(V_all, q_target, mode, mul_fn):
    """
    V_all: (N, 5, 4) normalized features for all 5 qubits.
    Constructs feature matrix for classifying qubit `q_target`:
      - 'single_lin2'     ( 2 feats): [a_k, b_k] (Standard single-qubit matched filter)
      - 'single_lin4'     ( 4 feats): [a_k, b_k, c_k, d_k] (Adds RMF, ignores neighbor qubits)
      - 'single_quad14'   (14 feats): single_lin4 + 10 self-quadratic terms (Catches T1 decay, ignores crosstalk)
      - 'multi_lin20'     (20 feats): All 20 linear features across the 5 qubits (Cancels linear crosstalk)
      - 'multi_cross_quad'(46 feats): All 20 linear features + 10 self-quadratic terms of qubit k
                                      + 16 inter-qubit quadratic cross-terms (a_k, b_k) x (a_j, b_j) for j != k!
    """
    vk = V_all[:, q_target, :]                                         # (N, 4)
    if mode == "single_lin2":
        return vk[:, :2]
    if mode == "single_lin4":
        return vk
    self_quad = [mul_fn(vk[:, i], vk[:, j]) for i in range(4) for j in range(i, 4)]  # 10 terms
    if mode == "single_quad14":
        return np.c_[vk, np.stack(self_quad, axis=1)]

    lin_all = V_all.reshape(len(V_all), N_QUBITS * 4)                  # 20 linear features
    if mode == "multi_lin20":
        return lin_all

    if mode == "multi_cross_quad":
        cross_terms = []
        for j in range(N_QUBITS):
            if j == q_target:
                continue
            vj = V_all[:, j, :]
            # 4 state-to-state cross products between target qubit k and neighbor qubit j
            cross_terms.extend([
                mul_fn(vk[:, 0], vj[:, 0]),
                mul_fn(vk[:, 0], vj[:, 1]),
                mul_fn(vk[:, 1], vj[:, 0]),
                mul_fn(vk[:, 1], vj[:, 1]),
            ])
        return np.c_[lin_all, np.stack(self_quad, axis=1), np.stack(cross_terms, axis=1)]

    raise ValueError(mode)


# ------------------------------------------------------------------ 4. Metrics: Fidelity, 5Q Geometric Mean, Joint 32-State, Cross-Fidelity
def best_threshold(s, y):
    order = np.argsort(s)
    ss, ys = s[order], y[order]
    n0 = (ys == 0).sum()
    c1 = np.r_[0, np.cumsum(ys == 1)]
    c0 = np.r_[0, np.cumsum(ys == 0)]
    k = int(np.argmin(c1 + (n0 - c0)))
    if k == 0:
        return ss[0] - 1.0
    if k == len(ss):
        return ss[-1]
    return 0.5 * (ss[k - 1] + ss[k])


def compute_cross_fidelity(Pred_5q, Y_true_5q):
    """
    Kent et al. (2025) Eq. 6 & HERQULES (2023) Sec. 4.3.3:
    F^{CF}_{j,k} = 1 - [P(pred_j=1 | true_k=0) + P(pred_j=0 | true_k=1)] for j != k.
    Lower |F^{CF}| means better crosstalk suppression!
    """
    cf_by_dist = {1: [], 2: [], 3: [], 4: []}
    for j in range(N_QUBITS):
        for k in range(N_QUBITS):
            if j == k:
                continue
            p1_given_0 = np.mean(Pred_5q[Y_true_5q[:, k] == 0, j] == 1)
            p0_given_1 = np.mean(Pred_5q[Y_true_5q[:, k] == 1, j] == 0)
            cf = abs(1.0 - (p1_given_0 + p0_given_1))
            cf_by_dist[abs(j - k)].append(cf)
    return {d: np.mean(vals) for d, vals in cf_by_dist.items()}, np.mean([v for vals in cf_by_dist.values() for v in vals])


# ------------------------------------------------------------------ 5. Run Float64 Benchmark Across Architectures
Acc_clf_f = accumulate_5q(I_clf, Q_clf, premixed_W)
Acc_te_f  = accumulate_5q(I_te,  Q_te,  premixed_W)

# Standardize base features on CLF
mu_f = Acc_clf_f.mean(axis=0)                                          # (5, 4)
A_f  = np.percentile(np.abs(Acc_clf_f - mu_f), FP_CFG["clip"], axis=0)
n_f  = np.ceil(np.log2(np.maximum(A_f, 1e-12)))

V_clf_f = np.clip((Acc_clf_f + 2.0 ** n_f - mu_f) / 2.0 ** (n_f + 1), 0.0, 1.0) - 0.5
V_te_f  = np.clip((Acc_te_f  + 2.0 ** n_f - mu_f) / 2.0 ** (n_f + 1), 0.0, 1.0) - 0.5

MODES = ["single_lin2", "single_lin4", "single_quad14", "multi_lin20", "multi_cross_quad"]
print("\n=== 5-Qubit Multiplexed Readout Benchmark (Float64 Reference) ===")
print(f"{'Architecture':<18s} {'#Mult/Q':>7s} | {'Q1':>6s} {'Q2':>6s} {'Q3':>6s} {'Q4':>6s} {'Q5':>6s} | {'F_5Q(GM)':>8s} {'Joint32':>8s} | {'<|F_CF|>':>8s}")
print("-" * 98)

float_results = {}
for mode in MODES:
    Pred_te = np.zeros_like(Y_te)
    fids = []
    n_mult = 0
    for q in range(N_QUBITS):
        Fc = np.c_[np.ones(len(V_clf_f)), build_multiqubit_features(V_clf_f, q, mode, lambda a, b: a * b)]
        Ft = np.c_[np.ones(len(V_te_f)),  build_multiqubit_features(V_te_f,  q, mode, lambda a, b: a * b)]
        n_mult = Fc.shape[1]
        t_target = 2.0 * Y_clf[:, q] - 1.0
        w = np.linalg.solve(Fc.T @ Fc + RIDGE_LAMBDA_REL * len(Fc) * np.eye(n_mult), Fc.T @ t_target)
        thr = best_threshold(Fc @ w, Y_clf[:, q])
        pred_q = (Ft @ w) > thr
        Pred_te[:, q] = pred_q.astype(int)
        f0 = np.mean(~pred_q[Y_te[:, q] == 0])
        f1 = np.mean(pred_q[Y_te[:, q] == 1])
        fids.append(0.5 * (f0 + f1))

    f5q_gm = float(np.prod(fids) ** (1.0 / N_QUBITS))
    joint_acc = float(np.mean(np.all(Pred_te == Y_te, axis=1)))
    cf_dist, cf_avg = compute_cross_fidelity(Pred_te, Y_te)
    float_results[mode] = (fids, f5q_gm, joint_acc, cf_dist, cf_avg, Pred_te)

    q_str = " ".join(f"{f:6.4f}" for f in fids)
    print(f"{mode:<18s} {n_mult:7d} | {q_str} | {f5q_gm:8.4f} {joint_acc:8.4f} | {cf_avg:8.5f}")


# ------------------------------------------------------------------ 6. Stage 3, 4 & 5: Bit-Accurate Fixed-Point 5-Qubit Hardware Model
print("\n=== Bit-Accurate Fixed-Point Hardware Verification (multi_cross_quad, 5 Qubits) ===")
lim_dlo = 2 ** (FP_CFG["dlo_bits"] - 1) - 1
W_dlo_int = np.zeros_like(premixed_W, dtype=np.int64)
for q in range(N_QUBITS):
    for f_idx in range(4):
        peak_w = np.abs(premixed_W[q, f_idx]).max()
        s = int(np.floor(np.log2(lim_dlo / max(peak_w, 1e-12))))
        W_dlo_int[q, f_idx] = np.clip(np.rint(premixed_W[q, f_idx] * (2.0 ** s)), -lim_dlo - 1, lim_dlo).astype(np.int64)

Acc_clf_int = np.rint(accumulate_5q(I_clf, Q_clf, W_dlo_int)).astype(np.int64)
Acc_te_int  = np.rint(accumulate_5q(I_te,  Q_te,  W_dlo_int)).astype(np.int64)

max_acc = int(max(np.abs(Acc_clf_int).max(), np.abs(Acc_te_int).max()))
acc_bits = int(np.ceil(np.log2(max_acc + 1))) + 1
print(f"Stage 2 5-Qubit DLO MAC peak accumulator requires: {acc_bits} signed bits (fits in 48-bit DSP48E2 P register)")

# Stage 3: 4-Cycle Bit-Shift Normalizer per (qubit, feature)
F_bits = FP_CFG["F"]
one64 = np.int64(1)
mu_int = np.rint(Acc_clf_int.mean(axis=0)).astype(np.int64)            # (5, 4)
A_int  = np.percentile(np.abs(Acc_clf_int - mu_int), FP_CFG["clip"], axis=0)
n_int  = np.ceil(np.log2(np.maximum(A_int, 1.0))).astype(np.int64)     # (5, 4)


def norm_5q_int(acc):
    tmp = acc + (one64 << n_int) - mu_int
    tmp = np.clip(tmp, 0, one64 << (n_int + 1))
    u = (tmp << F_bits) >> (n_int + 1)
    return u - (one64 << (F_bits - 1))                                 # 27-bit fixed-point in [-2^16, +2^16]


V_clf_int = norm_5q_int(Acc_clf_int)
V_te_int  = norm_5q_int(Acc_te_int)

# Stage 4 & 5: DSP48 Quadratic Expansion + 18-bit Ridge Dot Product + 1-Cycle Threshold Comparator
mul_dsp48 = lambda a, b: (a * b) >> F_bits
lim_w = 2 ** (FP_CFG["w_bits"] - 1)
w_frac = FP_CFG["w_frac"]

Pred_te_int = np.zeros_like(Y_te)
W_cls_5q = []
Thr_5q = []
fids_int = []

for q in range(N_QUBITS):
    Oc_int = np.c_[np.full(len(V_clf_int), 1 << F_bits, dtype=np.int64),
                   build_multiqubit_features(V_clf_int, q, "multi_cross_quad", mul_dsp48)]
    Ot_int = np.c_[np.full(len(V_te_int),  1 << F_bits, dtype=np.int64),
                   build_multiqubit_features(V_te_int,  q, "multi_cross_quad", mul_dsp48)]

    # Solve Ridge weights in float and auto-scale to 18-bit signed integer (`sfix18_12`)
    wf = np.linalg.solve(
        (Oc_int / 2.0**F_bits).T @ (Oc_int / 2.0**F_bits) + RIDGE_LAMBDA_REL * len(Oc_int) * np.eye(Oc_int.shape[1]),
        (Oc_int / 2.0**F_bits).T @ (2.0 * Y_clf[:, q] - 1.0)
    )
    rng_max = (lim_w - 1) / (2.0 ** w_frac)
    scale = 2.0 ** np.floor(np.log2(rng_max / np.abs(wf).max()))
    wq = np.clip(np.rint(wf * scale * (2.0 ** w_frac)), -lim_w, lim_w - 1).astype(np.int64)

    sc_c = Oc_int @ wq
    thr_q = int(round(best_threshold(sc_c.astype(np.float64), Y_clf[:, q])))
    sc_t = Ot_int @ wq
    pred_q = sc_t > thr_q

    W_cls_5q.append(wq)
    Thr_5q.append(thr_q)
    Pred_te_int[:, q] = pred_q.astype(int)
    f0 = np.mean(~pred_q[Y_te[:, q] == 0])
    f1 = np.mean(pred_q[Y_te[:, q] == 1])
    fids_int.append(0.5 * (f0 + f1))

W_cls_5q = np.stack(W_cls_5q, axis=0)                                  # (5, 47)
Thr_5q = np.array(Thr_5q, dtype=np.int64)                              # (5,)
f5q_int_gm = float(np.prod(fids_int) ** (1.0 / N_QUBITS))
joint_int  = float(np.mean(np.all(Pred_te_int == Y_te, axis=1)))
_, cf_int_avg = compute_cross_fidelity(Pred_te_int, Y_te)

f5q_flt_gm = float_results["multi_cross_quad"][1]
print(f"Float64 F_5Q (GM) = {f5q_flt_gm:.4f}  |  Fixed-Point (27b x 18b) F_5Q (GM) = {f5q_int_gm:.4f}  "
      f"(Delta = {100*(f5q_int_gm - f5q_flt_gm):+.3f} pts)")
print(f"Fixed-Point Joint 32-State Accuracy = {joint_int:.4f}  |  Mean Cross-Fidelity <|F_CF|> = {cf_int_avg:.5f}")

# Export 5-Qubit Golden Vectors for Vivado XSIM RTL Verification
np.savez(
    "golden_vectors_5q.npz",
    I_raw=I_te[:N_GOLD], Q_raw=Q_te[:N_GOLD], Y_true=Y_te[:N_GOLD],
    W_dlo_int=W_dlo_int.astype(np.int32),
    acc_5q=Acc_te_int[:N_GOLD], v_5q=V_te_int[:N_GOLD],
    mu_int=mu_int, n_int=n_int,
    W_cls_5q=W_cls_5q, Thr_5q=Thr_5q, Pred_5q=Pred_te_int[:N_GOLD].astype(np.int8),
)
print(f"Saved golden_vectors_5q.npz ({N_GOLD} 5-qubit multiplexed shots for Vivado RTL testbench).")

Mode 2: Constructing 5-Qubit Frequency-Multiplexed ADC Stream from QICK dataset ...
Synthesizing CAL (30000), CLF (50000), and TEST (20000) 5-qubit multiplexed shots ...
Multiplexed bus ready: I/Q shape=(20000, 500) (int16), Y_te shape=(20000, 5) (5 qubits)

Qubit 1 (IF= +64.73 MHz, T_opt=500): RMF caught 1258/14911 relaxing |1> shots (8.4%)
Qubit 2 (IF= +25.37 MHz, T_opt=500): RMF caught 1512/14999 relaxing |1> shots (10.1%)
Qubit 3 (IF= -24.79 MHz, T_opt=420): RMF caught 1509/15031 relaxing |1> shots (10.0%)
Qubit 4 (IF= -70.27 MHz, T_opt=480): RMF caught 1441/14856 relaxing |1> shots (9.7%)
Qubit 5 (IF=-127.28 MHz, T_opt=450): RMF caught 1415/14991 relaxing |1> shots (9.4%)

=== 5-Qubit Multiplexed Readout Benchmark (Float64 Reference) ===
Architecture       #Mult/Q |     Q1     Q2     Q3     Q4     Q5 | F_5Q(GM)  Joint32 | <|F_CF|>
--------------------------------------------------------------------------------------------------
single_lin2              3 | 0.8522 0.8201 0.8176 0.8